# R1000 Top30 Institutional Engine

Colab runbook for the current GitHub-based engine.

Flow:
1. Install dependencies
2. Mount Drive and load the project
3. Run collector
4. Run full pipeline
5. Run validation and inspect sleeve/rebalance outputs


In [ ]:
# 1) Pull latest GitHub master and mount Drive
from google.colab import drive
import os
import sys
import json
import importlib
import subprocess
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import time

def mount_drive_with_recovery(mountpoint='/content/drive'):
    try:
        drive.mount(mountpoint, force_remount=False)
        return
    except ValueError as exc:
        print(f'Initial Drive mount failed: {exc}')
        print('Retrying after unmount/cleanup ...')

    try:
        drive.flush_and_unmount()
    except Exception:
        pass

    subprocess.run(['bash', '-lc', 'fusermount -u /content/drive >/dev/null 2>&1 || true'], check=False)
    os.makedirs(mountpoint, exist_ok=True)
    time.sleep(2)

    try:
        drive.mount(mountpoint, force_remount=True)
    except ValueError as exc:
        raise ValueError(
            'Drive mount failed after recovery. Restart the Colab runtime, allow the Drive auth popup, '
            'and rerun this cell.'
        ) from exc

mount_drive_with_recovery('/content/drive')

BASE_DIR = '/content/drive/MyDrive/r1000_top30_institutional'
DATA_DIR = Path(BASE_DIR)
REPO_DIR = Path('/content/r1000-quant-engine')
REPO_URL = 'https://github.com/wscha231/r1000-quant-engine.git'
BRANCH = 'master'
FAST_MODE = True  # Use True for the first validation run, False for the final full run.

COMMON_CFG_OVERRIDES = {
    'sec_user_agent': 'R1000InstitutionalBot (contact: andrewcha231@gmail.com)',
    'fred_api_key': '8d92fb5a5de226657d912fe0284dfc00',
    'macro_refresh_days': 0,
    'live_refresh_days': 1,
    'companyfacts_refresh_days': 7,
    'alpha_vantage_free_refresh_tickers': 0,
    'alpha_vantage_free_statement_repair_tickers': 0,
    'alpha_vantage_free_statement_refresh_days': 7,
    'macro_slow_release_lag_months': 1,
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)

for name in ['r1000_top30_institutional', 'r1000_data_collector', 'r1000_portfolio_state', 'r1000_operator']:
    if name in sys.modules:
        del sys.modules[name]
importlib.invalidate_caches()

END_DATE = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d')

print('Repo path:', os.getcwd())
print('Python files:', [x.name for x in REPO_DIR.glob('*.py')])
print('Drive data path:', str(DATA_DIR))
print('End date:', END_DATE)
print('FAST_MODE:', FAST_MODE)


In [ ]:
# 2) Run collector
# - Keep the latest default sleeve/cash settings from GitHub master.
# - Refresh the collector outputs first.
from r1000_data_collector import collector_lean_full_run_cfg, run_data_collection

collector_cfg = collector_lean_full_run_cfg(BASE_DIR, end_date=END_DATE)
collector_cfg.update(COMMON_CFG_OVERRIDES)
collector_cfg['fast_mode'] = FAST_MODE

collector_summary = run_data_collection(collector_cfg)
print('=== collector output files ===')
print(collector_summary['output_files'])
print()
print('=== collector core coverage ===')
print(collector_summary['core_latest_coverage'])


In [ ]:
# 3) Run main pipeline + validation
# - Rebuild pipeline config from fresh defaults instead of copying the collector config.
from r1000_top30_institutional import run_default_pipeline
from r1000_data_collector import collector_lean_full_run_cfg, run_full_validation_suite

pipeline_cfg = collector_lean_full_run_cfg(BASE_DIR, end_date=END_DATE)
pipeline_cfg.update(COMMON_CFG_OVERRIDES)
pipeline_cfg['fast_mode'] = FAST_MODE
pipeline_cfg['reuse_existing_artifacts'] = True
pipeline_cfg['resume_partial_walkforward'] = False
pipeline_cfg['reuse_phase4_models_for_latest_recommendations'] = False
pipeline_cfg['force_full_fund_panel_rebuild'] = False

result = run_default_pipeline(pipeline_cfg)
report = run_full_validation_suite(pipeline_cfg, rerun_pipeline=False)

print('=== acceptance_checks ===')
print(result['acceptance_checks'])
print()
print('=== backtest_policy_snapshot ===')
print(report['backtest_policy_snapshot'])
print()
print('=== rebalance_interval_comparison_snapshot ===')
print(report['rebalance_interval_comparison_snapshot'])
print()
print('=== sleeve_policy_snapshot ===')
print(report['sleeve_policy_snapshot'])
print()
print('=== ops_tracking_snapshot ===')
print(report['ops_tracking_snapshot'])
print()
print('=== operator_snapshot ===')
print(report.get('operator_snapshot', {}))
print()
print('=== portfolio_shape ===')
print(report['portfolio_shape'])


In [ ]:
# 4) Inspect outputs
# - Separate model target outputs from live operator outputs.
import pandas as pd

OUT = DATA_DIR / 'outputs'
OPS = OUT / 'ops'
REP = OUT / 'reports'

weights = json.loads((OUT / 'weights_latest.json').read_text(encoding='utf-8'))
run_summary = json.loads((OUT / 'run_summary.json').read_text(encoding='utf-8'))
portfolio = pd.read_csv(OUT / 'portfolio_latest.csv')
top30 = pd.read_csv(OUT / 'top30_latest.csv')
operator_summary = json.loads((OPS / 'live_operator_summary.json').read_text(encoding='utf-8')) if (OPS / 'live_operator_summary.json').exists() else {}
operator_plan = pd.read_csv(OPS / 'live_operator_plan_latest.csv') if (OPS / 'live_operator_plan_latest.csv').exists() else pd.DataFrame()
live_state = json.loads((OPS / 'live_portfolio_state.json').read_text(encoding='utf-8')) if (OPS / 'live_portfolio_state.json').exists() else {}
rebalance = pd.read_csv(REP / 'rebalance_interval_comparison.csv') if (REP / 'rebalance_interval_comparison.csv').exists() else pd.DataFrame()

print('weights_latest sleeve targets:', weights.get('sleeve_target_weights'))
print('weights_latest sleeve actuals:', weights.get('sleeve_actual_weights'))
print('run_summary rebalance action:', run_summary.get('rebalance_action'))
print('run_summary active rebalance interval:', run_summary.get('active_rebalance_interval_months'))
print('operator summary:', operator_summary)
print('live state source:', live_state.get('state_source'))
print('live state positions:', len(live_state.get('positions', [])))

display(portfolio[[
    'ticker',
    'weight',
    'portfolio_sleeve_label',
    'portfolio_sleeve_confidence',
    'sage_sector',
    'sage_composite_score',
    'rebalance_action',
    'active_rebalance_interval_months',
]].head(20))

display(top30[[
    'rank',
    'ticker',
    'score',
    'portfolio_sleeve_label',
    'sage_composite_score',
    'portfolio_seed_score',
    'portfolio_alpha',
    'portfolio_utility',
]].head(30))

if not operator_plan.empty:
    display(operator_plan[[
        'decision_rank',
        'ticker',
        'current_weight',
        'target_weight_model',
        'recommended_weight',
        'weight_delta',
        'action',
        'action_reason',
        'held_days',
        'unrealized_return',
    ]].head(30))

display(rebalance)


In [ ]:
# 5) macro / feature sanity check
scored = pd.read_csv(OUT / 'scored_latest.csv')

macro_cols = [
    'm2_yoy_lag1m',
    'fed_assets_bil',
    'reverse_repo_bil',
    'tga_bil',
    'net_liquidity_bil',
    'net_liquidity_change_1m_bil',
    'liquidity_impulse_score',
    'liquidity_drain_score',
]

sage_cols = [
    'sage_sector',
    'sage_composite_score',
    'sage_g_score',
    'sage_v_score',
    'sage_q_score',
    'sage_c_score',
]

print('macro coverage:')
display(scored[[c for c in macro_cols if c in scored.columns]].notna().mean().sort_values(ascending=False))
print('sage coverage:')
display(scored[[c for c in sage_cols if c in scored.columns]].notna().mean().sort_values(ascending=False))


In [ ]:
scored = pd.read_csv(OUT / 'scored_latest.csv')

macro_cols = [
    'm2_yoy_lag1m',
    'fed_assets_bil',
    'reverse_repo_bil',
    'tga_bil',
    'net_liquidity_bil',
    'net_liquidity_change_1m_bil',
    'liquidity_impulse_score',
    'liquidity_drain_score',
]

display(scored[[c for c in macro_cols if c in scored.columns]].notna().mean().sort_values(ascending=False))
